### Impotração das Bibliotecas

In [1]:
# Importação das bibliotecas necessárias
# 
import os
import pandas as pd
from sqlalchemy import create_engine

# Configurações de Conexão com o PostgreSQL Definindo as variáveis de conexão
USUARIO = "postgres"
SENHA = "990957"  # <- Altere para a sua senha do PostgreSQL
HOST = "localhost"
PORTA = "5432"
BANCO = "supermarket_db"

conexao_url = f"postgresql://{USUARIO}:{SENHA}@{HOST}:{PORTA}/{BANCO}"

###Extraindo dados da Raw para utilizar como DataFrame com Pandas

In [2]:
engine = create_engine(conexao_url)
        
# 1. Leitura dos dados brutos diretamente da tabela raw_vendas no PostgreSQL
print("Lendo dados da tabela 'raw_vendas' no PostgreSQL...")
query = "SELECT * FROM raw_vendas;"
df = pd.read_sql(query, con=engine)

print(f"Registros lidos: {len(df)}")

df.head()

Lendo dados da tabela 'raw_vendas' no PostgreSQL...
Registros lidos: 1000


,invoice_id,branch,city,customer_type,gender,product_line,unit_price,quantity,tax_5_percent,Sales,date,time,payment,cogs,gross_margin_percentage,gross_income,rating
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,1:08:00 PM,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29:00 AM,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,1:23:00 PM,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,8:33:00 PM,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37:00 AM,Ewallet,604.17,4.761905,30.2085,5.3


In [6]:
#Comandos para informações dos tipos de dados
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   invoice_id               1000 non-null   str    
 1   branch                   1000 non-null   str    
 2   city                     1000 non-null   str    
 3   customer_type            1000 non-null   str    
 4   gender                   1000 non-null   str    
 5   product_line             1000 non-null   str    
 6   unit_price               1000 non-null   float64
 7   quantity                 1000 non-null   int64  
 8   tax_5_percent            1000 non-null   float64
 9   Sales                    1000 non-null   float64
 10  date                     1000 non-null   str    
 11  time                     1000 non-null   str    
 12  payment                  1000 non-null   str    
 13  cogs                     1000 non-null   float64
 14  gross_margin_percentage  1000 non-nu

In [ ]:
# Verificando linhas duplicadas no DataFrame - Não apresentou nenhuma linha
print(f"Linhas duplicadas: {df.duplicated().sum()}")

Linhas duplicadas: 0


In [ ]:
#Verificando valores nulos no DataFrame - Não apresentou nenhum valor nulo
print(df.isnull().sum())

invoice_id                 0
branch                     0
city                       0
customer_type              0
gender                     0
product_line               0
unit_price                 0
quantity                   0
tax_5_percent              0
Sales                      0
date                       0
time                       0
payment                    0
cogs                       0
gross_margin_percentage    0
gross_income               0
rating                     0
dtype: int64


In [26]:
df.isnull().sum()

invoice_id                 0
branch                     0
city                       0
customer_type              0
gender                     0
product_line               0
unit_price                 0
quantity                   0
tax_5_percent              0
Sales                      0
date                       0
time                       0
payment                    0
cogs                       0
gross_margin_percentage    0
gross_income               0
rating                     0
dtype: int64

In [14]:
#  Verificação e remoção de duplicidades


duplicados_antes = df.duplicated(subset=['invoice_id']).sum()
if duplicados_antes > 0:
    print(f"Removendo {duplicados_antes} registros duplicados...")
    df.drop_duplicates(subset=['invoice_id'], inplace=True)

    

In [15]:
 # Alteração do tipo de dado da coluna 'date' para datetime

df['date'] = pd.to_datetime(df['date'])

In [18]:
#  Tratamento de tipos de dados e conversões

#Cria variavel para uma lista de colunas que devem ser convertidas para numéricas
colunas_numericas = ['unit_price', 'quantity', 'tax_5_percent', 'Sales', 'cogs', 'gross_income', 'rating']

# Converte as colunas para numéricas, tratando erros com 'coerce' para evitar falhas na conversão
for col in colunas_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Tratamento de valores ausentes 
if df.isnull().sum().sum() > 0:
    print("Tratando valores ausentes...")
    df['rating'] = df['rating'].fillna(df['rating'].mean())
    df.dropna(subset=['invoice_id'], inplace=True)

In [ ]:
# Renomear colunas conforme o Dicionário de Dados

print("traduzindo nomes das colunas para deixar conforme o Dicionário de Dados...")
mapeamento_tratado = {
    'invoice_id': 'id_venda',
    'branch': 'Filial',
    'city': 'Cidade',
    'customer_type': 'tipo_cliente',
    'gender': 'Gênero',
    'product_line': 'linha_produto',
    'unit_price': 'preco_unitario',
    'quantity': 'Quantidade',
    'tax_5_percent': 'Imposto',
    'Sales': 'valor_total',  # <- CORREÇÃO AQUI TAMBÉM
    'date': 'data_venda',
    'time': 'hora_venda',
    'payment': 'forma_pagamento',
    'cogs': 'custo_mercadoria',
    'gross_margin_percentage': 'margem_percentual',
    'gross_income': 'receita_bruta',
    'rating': 'Avaliação'
}
df.rename(columns=mapeamento_tratado, inplace=True)

In [ ]:
 # 6. Exportação da Base Tratada para a camada Processed


pasta_destino = "data/processed"
os.makedirs(pasta_destino, exist_ok=True)
caminho_saida = os.path.join(pasta_destino, "vendas_tratadas.csv")

df.to_csv(caminho_saida, index=False, encoding='utf-8')

# Carregar para a tabela vendas_tratadas no banco
print("Inserindo dados na tabela 'vendas_tratadas' no PostgreSQL...")
df.to_sql('vendas_tratadas', con=engine, if_exists='replace', index=False)

print(f" ETL concluído com sucesso! Arquivo gerado em: {caminho_saida}")